*Figure notebook of our analysis (IDIBAPS cFos brains). It reads the project settings from the environment variable `CFOS_MAPPER_SETTINGS` (or `./settings.json`); set it before running:* `%env CFOS_MAPPER_SETTINGS=/path/to/settings.json`

# Step 1 - does cellfinder's pretrained network work on our data?

Kernel: **cellfinder-latest (1.11)**

For every brain we cut two small test regions (listed in `regions.json`: A = lateral cortex, B = centre of the brain), find bright cell-like spots ("candidates") and ask cellfinder's **pretrained** network which of them are cells.

Two checks:
- **Random spots (every brain):** 300 random positions per region. Those more than 10 um away from any candidate are plain tissue, so every one called "cell" is a **false positive on background**. A network that works should have close to 0%.
- **Manual annotations (where available):** cells found / missed, and false positives among annotated non-cells. Updated automatically as you annotate (`2_annotate.ipynb`).

Outputs: `results/<brain>/test_regions/step1_report.png` (one per brain), `results/step1_summary.csv` and `results/step1_summary.png`.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import napari
import cfos_mapper as cf

settings = cf.load_settings()
regions = cf.load_regions(settings)
brains = [b for b in regions if not b.startswith("_")]
results_folder = Path(settings["results_folder"])
print(brains)

## 1. Run on all brains (about 1-2 minutes per region)

Regions and candidates are made once and reused (the annotation notebook uses the same candidate files). `small_region=True` is only needed for regions cut out of a brain (see `find_candidates` in cellfinder_functions.py).

In [ ]:
for brain_name in brains:
    brain = cf.open_brain(brain_name, settings)
    folder = results_folder / brain_name / "test_regions"
    folder.mkdir(parents=True, exist_ok=True)

    for name, box in regions[brain_name].items():
        # cut the region out of the brain (only once)
        path = folder / f"{name}.ome.zarr"
        if not path.exists():
            cf.save_region(brain, box["z"], box["y"], box["x"], path)
        region = cf.mm.ZarrManager(str(path))
        signal = cf.load_into_memory(cf.get_channel(region, settings["signal_channel"]))
        background = cf.load_into_memory(cf.get_channel(region, settings["background_channel"]))
        size = cf.voxel_size(region)

        # candidates (found once, then read from file)
        candidates_file = folder / f"{name}_candidates.csv"
        if not candidates_file.exists():
            cf.find_candidates(signal, size, settings, small_region=True).to_csv(candidates_file, index=False)
        candidates = pd.read_csv(candidates_file)

        # pretrained network on the candidates
        result = cf.classify_candidates(candidates, signal, background, size, settings)
        result.to_csv(folder / f"{name}_pretrained.csv", index=False)

        # control: pretrained network on random spots
        random_spots = cf.random_positions(signal.shape, size, settings)
        control = cf.classify_candidates(random_spots, signal, background, size, settings)
        control.to_csv(folder / f"{name}_random_control.csv", index=False)

        print(f"{brain_name} {name}: {len(candidates)} candidates, "
              f"{result.is_cell.mean():.0%} called cell; random spots {control.is_cell.mean():.0%} called cell")

## 2. Statistics
Read from the files above, so this can be re-run any time (e.g. after more annotation) without re-running step 1.

In [ ]:
def percent(part, total):
    return round(100 * part / total, 1) if total else np.nan

rows = []
for brain_name in brains:
    folder = results_folder / brain_name / "test_regions"
    for name in regions[brain_name]:
        size = cf.voxel_size(cf.mm.ZarrManager(str(folder / f"{name}.ome.zarr")))
        candidates = pd.read_csv(folder / f"{name}_candidates.csv")
        result = pd.read_csv(folder / f"{name}_pretrained.csv")
        control = pd.read_csv(folder / f"{name}_random_control.csv")
        plain_tissue = cf.far_from_candidates(control, candidates, size)
        annotations = cf.load_annotations(folder / f"{name}_annotations.csv")
        a = cf.compare_with_annotations(result, annotations)
        n_cells = a["cells_found"] + a["cells_missed"]
        n_not_cells = a["false_positives"] + a["correctly_rejected"]

        rows.append({
            "brain": brain_name, "region": name,
            "candidates classified": len(result),
            "% called cell": percent(result.is_cell.sum(), len(result)),
            "plain-tissue spots": len(plain_tissue),
            "% false positives on plain tissue": percent(plain_tissue.is_cell.sum(), len(plain_tissue)),
            "annotated cells": n_cells,
            "% cells found": percent(a["cells_found"], n_cells),
            "cells missed": a["cells_missed"],
            "annotated non-cells": n_not_cells,
            "% false positives (annotated)": percent(a["false_positives"], n_not_cells),
        })

stats = pd.DataFrame(rows)
stats.to_csv(results_folder / "step1_summary.csv", index=False)
stats

## 3. Report per brain
Top: max projection of 30 cFos planes; yellow = called a cell, blue = called not a cell. Bottom: statistics. Saved as `results/<brain>/test_regions/step1_report.png`.

In [ ]:
table_columns = [c for c in stats.columns if c not in ("brain",)]

for brain_name in brains:
    folder = results_folder / brain_name / "test_regions"
    fig = plt.figure(figsize=(16, 11))
    grid = fig.add_gridspec(2, 2, height_ratios=[4, 1])
    for col, name in enumerate(regions[brain_name]):
        region = cf.mm.ZarrManager(str(folder / f"{name}.ome.zarr"))
        signal = cf.load_into_memory(cf.get_channel(region, settings["signal_channel"]))
        result = pd.read_csv(folder / f"{name}_pretrained.csv")
        cf.draw_calls(fig.add_subplot(grid[0, col]), signal, result, f"{brain_name} {name}")

    ax = fig.add_subplot(grid[1, :])
    ax.axis("off")
    brain_stats = stats[stats.brain == brain_name][table_columns].fillna("-")
    table = ax.table(cellText=brain_stats.values, colLabels=[c.replace(" ", "\n", 1) for c in table_columns],
                     loc="center", cellLoc="center")
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 2.6)
    fig.suptitle(f"{brain_name}: cellfinder pretrained network", fontsize=14)
    fig.tight_layout()
    fig.savefig(folder / "step1_report.png", dpi=70)
    plt.show()

## 4. Summary over all brains: why we need to retrain
Left: share of candidates called a cell. Right: **false positives on plain tissue** (should be close to 0%).

In [ ]:
labels = [f"{b}\n{r.split('_')[0][-1]}" for b, r in zip(stats.brain, stats.region)]
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, column in zip(axes, ["% called cell", "% false positives on plain tissue"]):
    ax.bar(labels, stats[column], color="gray")
    for i, v in enumerate(stats[column]):
        ax.text(i, v + 1, f"{v:.0f}%", ha="center")
    ax.set_title(f"pretrained network: {column}")
    ax.set_ylim(0, 105)
    ax.set_ylabel("%")
fig.tight_layout()
fig.savefig(results_folder / "step1_summary.png", dpi=80)
plt.show()

## 5. Look at one region in napari

In [ ]:
brain_name = "O15"
name = "regionB_central"

folder = results_folder / brain_name / "test_regions"
region = cf.mm.ZarrManager(str(folder / f"{name}.ome.zarr"))
result = pd.read_csv(folder / f"{name}_pretrained.csv")
size = cf.voxel_size(region)

viewer = napari.Viewer()
viewer.add_image(cf.get_channel(region, "auto"), name="autofluorescence",
                 scale=size, colormap="green", blending="additive", visible=False)
viewer.add_image(cf.get_channel(region, "cFos"), name="cFos",
                 scale=size, colormap="gray", blending="additive")

viewer.dims.thickness = (12.0, 0.0, 0.0)  # rings shrink on the planes above/below, like a sphere
points = dict(scale=size, size=10, symbol="ring", border_width=0,
              projection_mode="rescale_spherical", ndim=3)
viewer.add_points(result[~result.is_cell][["z", "y", "x"]].to_numpy(dtype=float),
                  name="not cell", face_color="deepskyblue", **points)
viewer.add_points(result[result.is_cell][["z", "y", "x"]].to_numpy(dtype=float),
                  name="cell", face_color="yellow", **points)